# Spotify Chart Data Analysis

**Project type:** Beginner data analyst portfolio project  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Dataset:** `spotify_cleaned(1).csv`

## Project objective
Explore Spotify chart records to understand chart positions, artist appearances, popularity scores, track duration, and patterns over time.

This notebook is designed to run from top to bottom. It checks the dataset before analysis and avoids assuming that repeated songs across different dates are duplicate records.

> **Note:** Results in this notebook are descriptive. Relationships between variables do not prove causation.


## 1. Import libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


ModuleNotFoundError: No module named 'matplotlib'

## 2. Load the dataset

Put this notebook in the same folder as `spotify_cleaned(1).csv`, or update `CSV_PATH` to the correct file path.


In [ ]:
from pathlib import Path

CSV_PATH = Path("spotify_cleaned(1).csv")
df = pd.read_csv(CSV_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 3. Understand the data

Check the dimensions, column names, data types, summary statistics, and missing values before drawing conclusions.


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nMissing values by column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nExact duplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T)


## 4. Prepare and validate columns

This section is safe to rerun: it standardizes column names and converts relevant fields when present. It does not automatically delete unusual values.


In [ ]:
# Standardize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\\s+", "_", regex=True)
)

# Trim surrounding whitespace in text columns
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

# Convert chart date if available
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Convert numeric columns if available
for col in ["position", "popularity", "duration_ms", "total_tracks", "release_year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Convert explicit flag carefully
if "is_explicit" in df.columns:
    if df["is_explicit"].dtype == "object" or str(df["is_explicit"].dtype) == "string":
        explicit_map = {
            "true": True, "false": False,
            "1": True, "0": False,
            "yes": True, "no": False
        }
        df["is_explicit"] = (
            df["is_explicit"].astype("string").str.lower().map(explicit_map).astype("boolean")
        )

print("Shape after type/format preparation:", df.shape)


## 5. Data quality checks

Review these results before deciding whether any records should be removed or corrected. A negative value or an unusual ranking may have a valid explanation in some datasets.


In [ ]:
# Missing values and duplicate rows
quality_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})
display(quality_report)

print("Exact duplicate rows:", df.duplicated().sum())

if "position" in df.columns:
    print("\nPosition summary:")
    display(df["position"].describe())
    print("Positions outside expected 1-50 range:",
          ((df["position"] < 1) | (df["position"] > 50)).sum())

if "popularity" in df.columns:
    print("\nPopularity summary:")
    display(df["popularity"].describe())
    print("Popularity outside expected 0-100 range:",
          ((df["popularity"] < 0) | (df["popularity"] > 100)).sum())

if "duration_ms" in df.columns:
    print("\nNon-positive track durations:", (df["duration_ms"] <= 0).sum())

if "date" in df.columns:
    print("\nInvalid chart dates:", df["date"].isna().sum())

if {"date", "song", "artist"}.issubset(df.columns):
    repeated_chart_entries = df.duplicated(subset=["date", "song", "artist"]).sum()
    print("Repeated date-song-artist combinations:", repeated_chart_entries)


## 6. Feature engineering

Create a duration-in-minutes column if duration is available. If the dataset already contains this column, it will be recalculated from milliseconds when possible.


In [ ]:
if "duration_ms" in df.columns:
    df["duration_minutes"] = (df["duration_ms"] / 60000).round(2)

display(df.head())


## 7. Exploratory analysis: chart position distribution

Lower position numbers indicate better chart rankings. This chart shows how the dataset's chart positions are distributed.


In [ ]:
if "position" in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df["position"].dropna(), bins=range(1, 52), discrete=True)
    plt.title("Distribution of Spotify Chart Positions")
    plt.xlabel("Chart position (lower is better)")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()
else:
    print("Column 'position' not found; skipping this chart.")


## 8. Most frequent songs and artists

These counts measure appearances in the dataset, not necessarily unique songs or unique artists. Artist collaborations stored in one cell are counted as one combined label.


In [ ]:
if "song" in df.columns:
    print("Top 10 songs by chart appearances:")
    display(df["song"].value_counts().head(10).to_frame("chart_appearances"))

if "artist" in df.columns:
    print("Top 10 artist labels by chart appearances:")
    artist_counts = df["artist"].value_counts().head(10).sort_values()
    plt.figure(figsize=(10, 6))
    artist_counts.plot(kind="barh")
    plt.title("Top 10 Artist Labels by Chart Appearances")
    plt.xlabel("Number of records")
    plt.ylabel("Artist label")
    plt.tight_layout()
    plt.show()


## 9. Popularity versus chart position

Explore whether popularity scores are associated with chart position. Since a smaller position is better, interpret the direction carefully.


In [ ]:
if {"popularity", "position"}.issubset(df.columns):
    plot_df = df[["popularity", "position"]].dropna()
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=plot_df, x="popularity", y="position", alpha=0.35, s=25)
    plt.title("Spotify Popularity vs. Chart Position")
    plt.xlabel("Popularity score")
    plt.ylabel("Chart position (lower is better)")
    plt.tight_layout()
    plt.show()

    corr = plot_df["popularity"].corr(plot_df["position"])
    print(f"Pearson correlation: {corr:.3f}")
    print("Correlation describes association; it does not establish causation.")
else:
    print("Required columns not found; skipping popularity analysis.")


## 10. Chart performance over time

The line below shows average chart position by date. Interpret this with care: the average can change when the mix of songs in the chart changes.


In [ ]:
if {"date", "position"}.issubset(df.columns):
    daily = (
        df.dropna(subset=["date", "position"])
          .groupby("date", as_index=False)["position"]
          .mean()
          .sort_values("date")
    )
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=daily, x="date", y="position")
    plt.title("Average Chart Position Over Time")
    plt.xlabel("Chart date")
    plt.ylabel("Average position")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not found; skipping time-series analysis.")


## 11. Explicit vs. non-explicit tracks

Compare chart-position distributions by explicit-content flag. This is descriptive only and does not show that explicit content causes a ranking outcome.


In [ ]:
if {"is_explicit", "position"}.issubset(df.columns):
    compare_df = df[["is_explicit", "position"]].dropna()
    if not compare_df.empty:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=compare_df, x="is_explicit", y="position")
        plt.title("Chart Position by Explicit-Content Flag")
        plt.xlabel("Explicit content")
        plt.ylabel("Chart position (lower is better)")
        plt.tight_layout()
        plt.show()

        display(
            compare_df.groupby("is_explicit")["position"]
            .agg(["count", "mean", "median"])
            .round(2)
        )
else:
    print("Required columns not found; skipping explicit-content comparison.")


## 12. Track duration analysis

Review the duration distribution and summary statistics. Extremely long or short durations should be investigated rather than removed automatically.


In [ ]:
if "duration_minutes" in df.columns:
    display(df["duration_minutes"].describe().to_frame("duration_minutes"))

    plt.figure(figsize=(9, 5))
    sns.histplot(df["duration_minutes"].dropna(), bins=30)
    plt.title("Distribution of Track Durations")
    plt.xlabel("Duration (minutes)")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.show()


## 13. Summarize findings

Run this section after the analysis cells. Use the computed values to write three to five evidence-based observations in your project README. Avoid making claims that are not supported by the results.


In [ ]:
print("DATASET SUMMARY")
print("=" * 40)
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")

if "date" in df.columns and df["date"].notna().any():
    print(f"Chart date range: {df['date'].min().date()} to {df['date'].max().date()}")

if "song" in df.columns:
    print(f"Distinct song labels: {df['song'].nunique(dropna=True):,}")

if "artist" in df.columns:
    print(f"Distinct artist labels: {df['artist'].nunique(dropna=True):,}")

if "popularity" in df.columns:
    print(f"Mean popularity: {df['popularity'].mean():.2f}")

if "position" in df.columns:
    print(f"Mean chart position: {df['position'].mean():.2f}")

print("\nNext: write your strongest verified findings in the README.md.")


## 14. Export an analysis-ready copy (optional)

This saves the prepared dataset in the notebook's current working folder. If you already have a cleaned CSV, keep the original file as well.


In [ ]:
OUTPUT_PATH = Path("spotify_analysis_ready.csv")
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved analysis-ready dataset to: {OUTPUT_PATH.resolve()}")
